In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [7]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','SibSp','Parch','Survived'])

In [9]:
df.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [13]:
df.isnull().sum()

Survived      0
Age         177
SibSp         0
Parch         0
Fare          0
dtype: int64

In [15]:
df.dropna(inplace=True)

In [17]:
df.isnull().sum()

Survived    0
Age         0
SibSp       0
Parch       0
Fare        0
dtype: int64

In [19]:
df['family'] = df['SibSp'] + df['Parch']
df.head()

,Survived,Age,SibSp,Parch,Fare,family
0,0,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,1
2,1,26.0,0,0,7.9250,0
3,1,35.0,1,0,53.1000,1
4,0,35.0,0,0,8.0500,0


In [25]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [27]:
df.head()

,Survived,Age,Fare,family
0,0,22.0,7.2500,1
1,1,38.0,71.2833,1
2,1,26.0,7.9250,0
3,1,35.0,53.1000,1
4,0,35.0,8.0500,0


In [31]:
x = df.iloc[:,1:]
y = df.iloc[:,0]

In [37]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [39]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


## Without binarization

In [44]:
clf = DecisionTreeClassifier()
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

accuracy_score(y_test,y_pred)

0.6223776223776224

In [48]:
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring='accuracy'))

0.6485524256651019

## Applying binarization

In [51]:
from sklearn.preprocessing import Binarizer

In [53]:
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [55]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [59]:
X_train_trf

array([[  1.    ,  31.    ,  20.525 ],
       [  1.    ,  26.    ,  14.4542],
       [  1.    ,  30.    ,  16.1   ],
       ...,
       [  0.    ,  41.    , 134.5   ],
       [  1.    ,  33.    ,  20.525 ],
       [  0.    ,  33.    ,   7.8958]])

In [61]:
X_test_trf

array([[  0.    ,  42.    ,  13.    ],
       [  1.    ,   3.    ,  18.75  ],
       [  1.    ,  29.    ,  26.    ],
       [  0.    ,  24.    ,  69.3   ],
       [  0.    ,  43.    ,   6.45  ],
       [  1.    ,   8.    ,  36.75  ],
       [  1.    ,  33.    ,  15.85  ],
       [  1.    ,  54.    ,  23.    ],
       [  0.    ,  28.    ,   7.8958],
       [  0.    ,  23.    ,   7.925 ],
       [  1.    ,  17.    ,  57.    ],
       [  0.    ,  32.5   ,  13.    ],
       [  1.    ,  31.    ,  57.    ],
       [  0.    ,  38.    ,   7.8958],
       [  0.    ,  35.    , 512.3292],
       [  1.    ,  23.    ,  11.5   ],
       [  0.    ,  23.5   ,   7.2292],
       [  1.    ,  16.    ,  57.9792],
       [  0.    ,  27.    ,  13.    ],
       [  1.    ,  54.    ,  59.4   ],
       [  1.    ,  16.    ,  46.9   ],
       [  1.    ,  50.    ,  26.    ],
       [  1.    ,   9.    ,  15.2458],
       [  0.    ,  35.    , 512.3292],
       [  0.    ,  22.    ,   9.35  ],
       [  0.    ,  17.   

In [63]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [73]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6293706293706294

In [83]:
X_trf = trf.fit_transform(x)

np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

0.6317683881064162